# ICD-10 Chapter Submission Pipeline

This notebook runs the same chapter/category implementation used in `src/icd10_coding/models/chapter.py`, starting from the raw CSV files and ending with a Kaggle-ready submission. The final model is a reference-augmented TF-IDF + LinearSVC classifier, with exact/fuzzy matching and ICD-description retrieval kept as fallback/detail signals.

## Setup

Import the libraries, define the project folders, and create the output directories used by the notebook.

In [1]:
from pathlib import Path
import re
import time
import unicodedata

import pandas as pd
from rapidfuzz import fuzz, process
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PREDICTIONS_DIR = PROJECT_ROOT / 'output' / 'predictions'
SUBMISSIONS_DIR = PROJECT_ROOT / 'output' / 'submissions'

for path in [DATA_PROCESSED_DIR, PREDICTIONS_DIR, SUBMISSIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## Helper Functions

These functions clean text, normalize accents/case, build the TF-IDF feature representation, and define the fallback order for the final chapter prediction.

In [2]:
def basic_clean(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = text.replace('`', "'").replace('??', "'")
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def normalize_text(text):
    text = basic_clean(text).lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    return text.strip()


def build_features():
    return FeatureUnion([
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ])


def first_char(value):
    if pd.isna(value) or value == '':
        return ''
    return str(value)[0]


def predict_chapter(row):
    if row['chapter_pred']:
        return row['chapter_pred']
    if row['s1_chapter']:
        return row['s1_chapter']
    if row['retrieval_chapter']:
        return row['retrieval_chapter']
    return 'null' 

## Load And Preprocess Data

Read the raw Kaggle files, create cleaned and normalized text columns, derive the chapter target from the first code character, and save processed CSVs for reuse.

In [3]:
train = pd.read_csv(DATA_RAW_DIR / 'codification_data.csv')
test = pd.read_csv(DATA_RAW_DIR / 'leaderboard_data.csv')
reference = pd.read_csv(DATA_RAW_DIR / 'icd_d_p_pairs.csv')

train['Literal_clean'] = train['Literal'].apply(basic_clean)
train['Literal_norm'] = train['Literal'].apply(normalize_text)
test['Literal_clean'] = test['Literal'].apply(basic_clean)
test['Literal_norm'] = test['Literal'].apply(normalize_text)
reference['Description_norm'] = reference['Description'].apply(normalize_text)

train['code_type'] = train['Code'].astype(str).str[0].apply(lambda c: 'diagnosis' if c.isalpha() else 'procedure')
train['chapter'] = train['Code'].astype(str).str[0]
reference['chapter'] = reference['Code'].astype(str).str[0]
train['is_icd10'] = train['Code'].isin(set(reference['Code']))
train = train.merge(reference[['Code', 'D_P', 'Description']], on='Code', how='left')

train.to_csv(DATA_PROCESSED_DIR / 'train_preprocessed.csv', index=False)
test.to_csv(DATA_PROCESSED_DIR / 'test_preprocessed.csv', index=False)
reference.to_csv(DATA_PROCESSED_DIR / 'reference_preprocessed.csv', index=False)

print(f'Training rows:  {len(train):,}')
print(f'Test rows:      {len(test):,}')
print(f'Reference rows: {len(reference):,}')

Training rows:  13,700
Test rows:      6,667
Reference rows: 179,742


## Train The Chapter Classifier

Train the main model: TF-IDF character/word features plus LinearSVC. The model learns from both training literals and official ICD descriptions, which gives it more labelled examples per chapter.

In [4]:
start = time.time()
vec_chapter = build_features()
chapter_texts = pd.concat([train['Literal_norm'], reference['Description_norm']], ignore_index=True)
chapter_labels = pd.concat([train['chapter'], reference['chapter']], ignore_index=True)

x_chapter = vec_chapter.fit_transform(chapter_texts)
clf_chapter = LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_STATE)
clf_chapter.fit(x_chapter, chapter_labels.values)

x_test = vec_chapter.transform(test['Literal_norm'])
decisions = clf_chapter.decision_function(x_test)
test['chapter_pred'] = clf_chapter.classes_[decisions.argmax(axis=1)]
test['chapter_confidence'] = decisions.max(axis=1)

print(f'Trained final chapter classifier in {time.time() - start:.1f}s')

Trained final chapter classifier in 149.3s


## Exact And Fuzzy Matching Signals

Build a lookup from normalized training literals to their most common full code. Exact matches catch repeated literals, while fuzzy matching catches near-duplicates and small spelling differences.

In [5]:
exact_lookup = {}
for lit_norm, group in train.groupby('Literal_norm'):
    exact_lookup[lit_norm] = group['Code'].value_counts().index[0]

test['step1_code'] = test['Literal_norm'].map(exact_lookup)
test['step1_method'] = test['step1_code'].apply(lambda x: 'exact' if pd.notna(x) else 'unresolved')

keys = list(exact_lookup.keys())
for idx, row in test[test['step1_code'].isna()].iterrows():
    result = process.extractOne(row['Literal_norm'], keys, scorer=fuzz.ratio, score_cutoff=60)
    if result is not None:
        test.loc[idx, 'step1_code'] = exact_lookup[result[0]]
        test.loc[idx, 'step1_method'] = 'fuzzy'

test['s1_chapter'] = test['step1_code'].apply(first_char)
print(test['step1_method'].value_counts())

step1_method
exact         3812
fuzzy         2819
unresolved      36
Name: count, dtype: int64


## ICD Description Retrieval Signal

Build TF-IDF indexes over official ICD descriptions and retrieve the closest reference code for each test literal. This provides a dictionary-style fallback signal.

In [6]:
reference['Description_norm'] = reference['Description_norm'].fillna('')
rcv = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_df=0.95, sublinear_tf=True)
rwv = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
rcr = rcv.fit_transform(reference['Description_norm'])
rwr = rwv.fit_transform(reference['Description_norm'])

vc = rcv.transform(test['Literal_norm'].fillna(''))
vw = rwv.transform(test['Literal_norm'].fillna(''))

retrieval_codes = []
retrieval_scores = []
for i in range(0, len(test), 100):
    end = min(i + 100, len(test))
    comb = (cosine_similarity(vc[i:end], rcr) + cosine_similarity(vw[i:end], rwr)) / 2
    best = comb.argmax(axis=1)
    for j in range(end - i):
        bi = best[j]
        retrieval_codes.append(reference.iloc[bi]['Code'])
        retrieval_scores.append(float(comb[j, bi]))
    del comb

test['retrieval_code'] = retrieval_codes
test['retrieval_score'] = retrieval_scores
test['retrieval_chapter'] = test['retrieval_code'].apply(first_char)

## Final Chapter Prediction

Choose the final `y_category`. In this implementation the direct chapter classifier is the primary prediction; matching and retrieval are fallback signals if needed.

In [7]:
test['y_category'] = test.apply(predict_chapter, axis=1)
print(test['y_category'].value_counts().head(15))

y_category
O    908
Z    856
0    625
B    308
N    304
E    260
K    250
I    233
3    216
R    213
D    208
F    183
M    166
6    160
4    145
Name: count, dtype: int64


## Validation Check

Repeat the same logic on an 80/20 train-validation split to estimate local chapter accuracy before submitting to Kaggle.

In [8]:
tr_split, val_split = train_test_split(train, test_size=0.2, random_state=RANDOM_STATE)
tr_split = tr_split.copy()
val_split = val_split.copy()
tr_split['Literal_norm'] = tr_split['Literal_norm'].fillna('')
val_split['Literal_norm'] = val_split['Literal_norm'].fillna('')
val_split['true_chapter'] = val_split['Code'].astype(str).str[0]

v_vec = build_features()
v_texts = pd.concat([tr_split['Literal_norm'], reference['Description_norm']], ignore_index=True)
v_labels = pd.concat([tr_split['chapter'], reference['chapter']], ignore_index=True)
v_x = v_vec.fit_transform(v_texts)
v_clf = LinearSVC(C=1.0, max_iter=3000, random_state=RANDOM_STATE)
v_clf.fit(v_x, v_labels.values)

v_decisions = v_clf.decision_function(v_vec.transform(val_split['Literal_norm']))
val_split['chapter_pred'] = v_clf.classes_[v_decisions.argmax(axis=1)]
val_split['chapter_confidence'] = v_decisions.max(axis=1)

v_lookup = {}
for lit_norm, group in tr_split.groupby('Literal_norm'):
    v_lookup[lit_norm] = group['Code'].value_counts().index[0]
val_split['step1_code'] = val_split['Literal_norm'].map(v_lookup)
val_split['step1_method'] = val_split['step1_code'].apply(lambda x: 'exact' if pd.notna(x) else 'unresolved')

v_keys = list(v_lookup.keys())
for idx, row in val_split[val_split['step1_code'].isna()].iterrows():
    result = process.extractOne(row['Literal_norm'], v_keys, scorer=fuzz.ratio, score_cutoff=60)
    if result is not None:
        val_split.loc[idx, 'step1_code'] = v_lookup[result[0]]
        val_split.loc[idx, 'step1_method'] = 'fuzzy'
val_split['s1_chapter'] = val_split['step1_code'].apply(first_char)

vc_val = rcv.transform(val_split['Literal_norm'])
vw_val = rwv.transform(val_split['Literal_norm'])
ret_codes = []
for i in range(0, len(val_split), 100):
    end = min(i + 100, len(val_split))
    comb = (cosine_similarity(vc_val[i:end], rcr) + cosine_similarity(vw_val[i:end], rwr)) / 2
    best = comb.argmax(axis=1)
    for j in range(end - i):
        ret_codes.append(reference.iloc[best[j]]['Code'])
    del comb
val_split['retrieval_code'] = ret_codes
val_split['retrieval_chapter'] = val_split['retrieval_code'].apply(first_char)
val_split['y_category'] = val_split.apply(predict_chapter, axis=1)

direct_acc = (val_split['chapter_pred'] == val_split['true_chapter']).mean()
combined_acc = (val_split['y_category'] == val_split['true_chapter']).mean()
print(f'Direct chapter classifier accuracy: {direct_acc * 100:.2f}%')
print(f'Combined chapter accuracy: {combined_acc * 100:.2f}%')

Direct chapter classifier accuracy: 55.88%
Combined chapter accuracy: 55.88%


## Save Submission

Write the Kaggle-ready chapter submission and a detailed prediction file with the intermediate signals for inspection.

In [9]:
submission = test[['id', 'Literal', 'y_category']].copy()
submission['y_category'] = submission['y_category'].fillna('null')
submission.loc[submission['y_category'] == '', 'y_category'] = 'null'
submission.to_csv(SUBMISSIONS_DIR / 'shareable_chapter_submission.csv', index=False)

detail = test[[
    'id', 'Literal', 'Literal_norm',
    'step1_code', 'step1_method', 's1_chapter',
    'retrieval_code', 'retrieval_score', 'retrieval_chapter',
    'chapter_pred', 'chapter_confidence', 'y_category',
]].copy()
detail.to_csv(PREDICTIONS_DIR / 'shareable_chapter_predictions.csv', index=False)

print(SUBMISSIONS_DIR / 'shareable_chapter_submission.csv')
print(PREDICTIONS_DIR / 'shareable_chapter_predictions.csv')
print(submission.shape)
submission.head()

d:\Users\herme\UNI\2\Natural language\project\project\output\submissions\shareable_chapter_submission.csv
d:\Users\herme\UNI\2\Natural language\project\project\output\predictions\shareable_chapter_predictions.csv
(6667, 3)


,id,Literal,y_category
0,1,AMNIODRENAJE,7
1,2,Hiperparatiroidismo primario,E
2,3,MIGRANYA parto,O
3,4,VHC,B
4,5,Absceso mama izq,N
